# Cross-lingual transfer — surface baseline (LP4FM)

**CPU runtime. Runtime → Run all.** No model is loaded, so this needs no GPU
and takes about ten minutes.

The results are already committed at `results/lp4fm/` — run this only to
reproduce them, add a role, or add a language.

**What it measures.** A character n-gram classifier is fitted on one
language and evaluated on another, on the same problems (matched by
`problem_id`), with the variable name masked out. It is the control the
cross-lingual probe result has never had: if a model-free baseline transfers
as well as the probe, transfer is surface regularity rather than a
language-universal role representation.

**Why only three languages.** `problem_id` is a hash of the problem
description, so it is stable across languages — but only Python, JavaScript
and PHP share ids at usable rates (2953 / 1529 / 1145 pairwise). Every other
XLCoST pair shares 11–175 of ~9000, so matched transfer is unavailable there
and `--matched` refuses rather than quietly comparing different algorithms.


In [ ]:
# 1 - setup. No torch, no transformers: nothing here loads a model.
import pathlib
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B main origin/main && git pull -q
!git log --oneline -1
!pip install -q numpy scikit-learn matplotlib tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" \
  "tree-sitter-java>=0.23.5" "tree-sitter-cpp>=0.23.4" \
  "tree-sitter-go>=0.25.0" "tree-sitter-ruby>=0.23.1"
print("setup complete")


In [ ]:
# 2 - CONFIG
ROLES = "accumulator iterator index_key"   # space-separated; any of the five
print(f"roles: {ROLES}")


In [ ]:
# 3 - the matrix. Builds corpora, extracts all roles, runs every ordered
#     pair, exports to results/lp4fm/. ~10 min, CPU.
!bash scripts/run_crosslang.sh {ROLES}


In [ ]:
# 4 - results
from IPython.display import Image, display
import glob
print(open("results/lp4fm/SUMMARY.md").read())
for f in sorted(glob.glob("results/lp4fm/heatmap_*.png")):
    print(f); display(Image(f))


In [ ]:
# 5 - commit to GitHub (optional).
#     Needs a Colab secret GH_TOKEN: fine-grained PAT, this repo only,
#     Contents = Read and write. Not a GPG key -- that signs commits, it
#     cannot authenticate a push. No token? Download results/lp4fm/ from the
#     file browser and commit from your local clone.
from google.colab import userdata
import os
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    print("No GH_TOKEN secret - download results/lp4fm/ and commit locally.")
else:
    os.environ["GH_TOKEN"] = tok
    !git config user.email "naingoolwin.astrio@gmail.com"
    !git config user.name "naingoolwin"
    !git add results/lp4fm
    !git commit -q -m "Cross-lingual surface baseline: {ROLES}" || echo "nothing to commit"
    !git push -q https://$GH_TOKEN@github.com/nolanlwin/mech-interp.git HEAD:main && echo "pushed"
